# 01b — Literature Review

**Course:** AML-DL Project — Phase 3 (Hybrid ML + DL Intrusion Detection)

## 1. Scope and rationale

This notebook situates the two Phase-3 hybrid architectures inside the existing literature on intrusion detection and anomaly detection. It is not a survey — it is a tightly scoped review of **four anchor papers**, two for each model, chosen because:

| Selection criterion | Why it matters for this submission |
|---|---|
| One *foundational* paper per model | Establishes the algorithmic primitive we rely on (Isolation Forest for Model A, XGBoost for Model C). Citing the foundational paper is the standard academic move when a method is named after it. |
| One *direct precedent* paper per model | Shows the most architecturally similar prior work to our hybrid. This grounds the "Hybrid Innovation" claim in the rubric — we can point to what was done before and articulate exactly what we change. |
| Top-tier venues only | All four papers are from ICDM, IEEE TKDE, IEEE TETCI, and ACM KDD — the standard outlets for IDS / anomaly-detection methodology. |
| At least one paper per model evaluated on NSL-KDD or its predecessor | Keeps the comparison apples-to-apples: prior numbers and our numbers can be discussed in the same paragraph. |

The four anchor papers are listed in the reference list at the bottom (Section 8). Section 2 is a brief background to anchor the vocabulary; Sections 3–4 walk through each model's two papers; Section 5 places the four papers on a single comparison table; Section 6 articulates the gap Phase 3 fills; Section 7 reads each paper through the lens of the Phase-3 rubric.

## 2. What this notebook is *not*

* Not a complete survey of the IDS literature — surveys exist (Buczak & Guven 2016, Chandola, Banerjee & Kumar 2009) and are cited in the reference list for context but not analysed cell-by-cell.
* Not a re-statement of Phase-1 / Phase-2 references — those notebooks already cite Tavallaee et al. (2009) for NSL-KDD and Sakurada & Yairi (2014) for autoencoder-based anomaly detection.
* Not a benchmark replication — we do not re-run any of the four anchor papers' code. We compare results from their tables to ours where the same dataset and metric are used.

## 3. Background — anomaly detection paradigms in IDS

Network intrusion detection systems fall along two orthogonal axes:

**Detection paradigm.**
* *Signature-based* (Snort, Suricata, classical Zeek rules) match against a hand-curated database of known attack patterns. Excellent precision on known attacks; effectively zero recall on novel ones.
* *Anomaly-based* learns a model of *normal* behaviour and flags deviations. Trades precision for novelty coverage. This is the paradigm we use across all three project phases.

**Supervision regime.** (see Chandola, Banerjee & Kumar 2009 for the canonical taxonomy)
* *Supervised* requires labeled examples of every class — usable for known-attack classification but fails on zero-day attacks.
* *Semi-supervised* / *one-class* trains only on normal traffic and flags anything that does not fit the learned manifold. This is what Phase 1 (Isolation Forest, Z-Score) and Phase 2 (Autoencoder, β-VAE) do.
* *Unsupervised* trains on unlabelled mixed traffic, assuming attacks are rare. Theoretically attractive but in practice contaminates the "normal" model.

**The IDS-specific challenges that shape our paper choices:**
* *Severe class imbalance* — `R2L` and `U2R` together are < 1 % of NSL-KDD train rows (Tavallaee et al. 2009). This penalises any method that optimises pure accuracy.
* *Distribution shift* — NSL-KDD test contains 17 attack types not present in train. This is what makes one-class detectors competitive: they do not need to have seen the attack to flag it.
* *Calibration* — IDS deployments cannot tolerate high false-positive rates. ROC-AUC is informative but not enough; precision at the chosen operating threshold is what determines analyst trust.

Each of the four anchor papers below addresses at least two of these three challenges. Phase 3's hybrids inherit the same vocabulary.

## 4. Foundations of Model A — Deep Isolation Forest

### 4.1 Liu, Ting & Zhou (2008) — *Isolation Forest*
Published at **IEEE ICDM 2008**, with an extended journal version in *ACM TKDD 2012*.

**Problem setup.** Existing distance-based and density-based anomaly detectors (k-NN distance, LOF, one-class SVM) scale poorly with sample size and dimensionality, and they rely on an explicit *normal-profile* model. The authors flip the framing: anomalies are *few* and *different*, which means they are also *easier to isolate*. So instead of modelling normality, they model isolation difficulty.

**Method.**
1. Build an ensemble of random binary trees (`iTrees`) over a subsample of the data.
2. For each split, pick a random feature and a random split value between the feature's min and max in the current node.
3. Continue until every leaf contains a single point or until `max_depth` is reached.
4. The anomaly score for a point `x` is its expected path length over the ensemble, normalised by the average path length of an unsuccessful BST search:
   $$ s(x) = 2^{-\,\mathbb{E}[h(x)] / c(n)} \quad \text{where} \quad c(n) = 2(\ln(n-1)+0.5772) - \frac{2(n-1)}{n} $$
   `s(x)` close to 1 ⇒ anomalous; close to 0.5 ⇒ normal-ish; close to 0 ⇒ very normal.

**Why it works.** A point in a sparse region of the input space is, on average, separated by a single random split; a point inside a dense cluster requires many splits. This works without any distance metric, scales linearly in `n`, and is embarrassingly parallel.

**What we adopt verbatim in Phase 3.**
* The path-length scoring formula and the `c(n)` normalisation, via `sklearn.ensemble.IsolationForest`.
* The recommended `max_samples = 256`, `n_estimators = 200` defaults.
* The convention that score-orientation depends on caller context — sklearn returns *higher = more normal*; we negate to get *higher = more anomalous*, so it matches the AE recon-error convention used elsewhere in the repository.

**What we change.** We do not feed the raw 43-D NSL-KDD features into IF. We feed the 32-D AE latent. This addresses the well-known weakness of IF on heterogeneous tabular data: random axis-aligned splits waste depth on highly correlated columns (`serror_rate` ≈ `srv_serror_rate`, etc.). The autoencoder's encoder learns a decorrelated, dense, lower-dimensional representation in which axis-aligned isolation is more efficient.

**Why we cite this paper specifically.** It is the algorithmic primitive of the ML half of Model A. Without it, the Phase-3 hybrid has no decision rule.

---

### 4.2 Xu, Pang, Wang & Wang (2023) — *Deep Isolation Forest for Anomaly Detection*
Published in **IEEE TKDE 2023**.

**Problem setup.** Liu et al.'s Isolation Forest plateaus on tabular data with non-trivial cross-feature correlations (the precise NSL-KDD failure mode discussed above) and on high-dimensional / unstructured data (images, text, time series) where axis-aligned splits are even worse. The authors propose a *generic* fix: replace the random axis-aligned splits with random *projections* learned by a neural network, so each iTree splits in a learned subspace rather than on a single raw feature.

**Method.** Each iTree is preceded by a small randomly initialised MLP that produces a learned representation `z = f(x)` of the input. Splits then operate on `z` rather than on `x`. Multiple MLPs (one per tree, or shared per ensemble) give the ensemble representational diversity. The path-length scoring formula is unchanged.

**Reported results.** Across 30+ tabular and unstructured benchmarks the authors report consistent F1 / AUC gains over vanilla Isolation Forest, with the largest gains on datasets where the raw features are highly correlated or where the anomaly signature is non-axis-aligned in the input space.

**Why we cite this paper specifically.** It is the direct paper of inspiration for Model A. Our architecture differs in a small but defensible way: instead of using random untrained MLPs to project the input (Xu et al.'s ensemble-style approach), we use a *single trained autoencoder* whose representation is learned via reconstruction loss on the normal-traffic distribution. This trades ensemble diversity in projection for representation quality, which is the right trade-off when the dataset is moderate in size (≈ 67 k normal training rows in NSL-KDD) and the normal manifold is well-defined.

**Adoption summary for Model A.**
| Element | Source paper | Adopted in Phase 3 |
|---|---|---|
| Path-length anomaly score | Liu 2008 | Verbatim, via sklearn |
| Hyperparameter defaults (`max_samples=256`, `n_estimators=200`) | Liu 2008 | `n_estimators=200` retained; `max_samples='auto'` (= min(256, n)) |
| IF on a learned representation | Xu 2023 | Adopted, with a single trained AE replacing per-tree random MLPs |
| Score-direction (higher = more anomalous) | Convention | Adopted (sklearn negation) |

## 5. Foundations of Model C — Cascaded AE + XGBoost

### 5.1 Shone, Ngoc, Phai & Shi (2018) — *A Deep Learning Approach to Network Intrusion Detection*
Published in **IEEE Transactions on Emerging Topics in Computational Intelligence** 2(1), 41–50.

**Problem setup.** Existing IDS classifiers on NSL-KDD reached ≈ 80 % multi-class accuracy but with poor per-class recall on `R2L` and `U2R`. The authors hypothesise that the bottleneck is feature representation rather than classifier capacity, and propose a non-symmetric deep autoencoder (NDAE) that learns a compressed representation specifically suited to subsequent supervised classification.

**Method.**
1. Train a stack of NDAEs in a layer-wise greedy regime on labeled NSL-KDD train data. Each NDAE is intentionally non-symmetric — the decoder is shallower than the encoder — to bias the representation toward classification-relevant features rather than reconstruction-perfect features.
2. Use the final encoder as a fixed feature extractor.
3. Train a Random Forest classifier on the encoded features for multi-class classification (`Normal`, `DoS`, `Probe`, `R2L`, `U2R`).

**Reported results.** On NSL-KDD test, the NDAE + RF model reaches ≈ 85.4 % accuracy and is competitive on per-class F1 with deeper end-to-end neural classifiers — at a fraction of the parameter count.

**Why we cite this paper specifically.** It is the closest architectural precedent for Model C: a deep autoencoder used as a feature extractor for a supervised tree ensemble, on the same dataset, evaluated with the same multi-class taxonomy. The architectural pattern (DL → ML) is essentially identical.

**What we change relative to Shone et al.**
* **Symmetric AE with skip connections** instead of stacked non-symmetric AEs. We do not need the asymmetry trick because we *also* feed the residual block (which Shone et al. do not), and because skip connections preserve fine-grained reconstruction information that the residuals can then expose to the classifier.
* **XGBoost instead of Random Forest.** Random Forest's bagging gives variance reduction but not bias reduction; gradient boosting fits successive trees to the residuals of the ensemble and is empirically stronger on the tabular IDS regime (see Chen & Guestrin 2016 below).
* **Three-block feature concatenation.** Shone et al. feed *only* the encoded features. We feed `[raw ‖ z ‖ |x − x̂|]`. The residual block is the architectural delta: it is not derivable by either component alone, and our notebook-04 SHAP analysis verifies that XGBoost actually uses it.
* **Leakage-safe split protocol.** We train the AE on a strict 80 % normal subset that the XGBoost classifier never sees, eliminating the optimistic-residuals problem (a row that the AE was trained on will, by construction, have unrealistically small residuals).

---

### 5.2 Chen & Guestrin (2016) — *XGBoost: A Scalable Tree Boosting System*
Published at **ACM KDD 2016**.

**Problem setup.** Gradient boosting (Friedman 2001) was already understood to dominate tabular benchmarks, but existing implementations (scikit-learn's `GradientBoostingClassifier`, R's `gbm`) were slow and memory-hungry. The authors propose a system-level redesign — sparse-aware split finding, parallel histogram construction, regularised objective, hardware-aware cache pre-fetching — that makes gradient boosting tractable at the dataset scales encountered in industry.

**Method.** For our purposes the relevant contributions are:
1. **Regularised objective.** XGBoost adds an L2 penalty on leaf weights and a complexity penalty proportional to the number of leaves. This stabilises training on small minority classes (R2L, U2R) where Random Forest tends to over-fit.
2. **Histogram-based split finding.** The `tree_method='hist'` setting we use bins continuous features into 256 buckets and finds the best split inside that compressed representation. Empirically ~10× faster than exact greedy split finding with negligible accuracy loss on NSL-KDD-scale data.
3. **`multi:softprob` objective.** Returns calibrated multi-class probabilities directly, which is exactly what the Streamlit demo shows the analyst, and what we feed into the binary projection `1 − P(Normal)` for cross-phase ROC comparison.

**Why we cite this paper specifically.** XGBoost is the algorithmic primitive of the ML half of Model C. Our hyperparameter choices (`n_estimators=400`, `max_depth=6`, `learning_rate=0.1`, `tree_method='hist'`, `multi:softprob`) are standard defaults from the paper's recommendations.

**Why XGBoost specifically (vs Random Forest as Shone et al. used).** Three reasons:
1. *Probabilistic output.* Random Forest's class-vote probabilities are notoriously poorly calibrated; XGBoost's `softprob` is much closer to true posterior. Calibration matters for the cascade gate's threshold tuning and for the demo's confidence display.
2. *Imbalance handling.* Boosting's residual-fitting structure naturally up-weights misclassified minority-class rows in subsequent trees. Random Forest treats all rows equivalently per tree.
3. *Standard tabular benchmark dominance.* On Kaggle and the OpenML AutoML benchmarks, XGBoost / LightGBM consistently top the leaderboard for tabular classification. Choosing it over Random Forest is the obvious modern call.

**Adoption summary for Model C.**
| Element | Source paper | Adopted in Phase 3 |
|---|---|---|
| AE → tree ensemble pattern on NSL-KDD | Shone 2018 | Adopted |
| Multi-class classification into `{Normal, DoS, Probe, R2L, U2R}` | Shone 2018 | Adopted verbatim |
| `multi:softprob` objective + histogram trees | Chen & Guestrin 2016 | Adopted |
| Symmetric AE with skip connections | Phase-2 work, this repo | Replaces Shone's NDAE |
| Three-block feature concatenation `[raw ‖ z ‖ residual]` | **Novel to Phase 3** | Architectural delta |
| Stage-1 AE gate for cascade short-circuit | **Novel to Phase 3** | Architectural delta |

## 6. Where Phase 3 sits — comparison table

Quick at-a-glance positioning of the four anchor papers and Phase 3.

| Work | Year | Venue | Dataset | DL component | ML component | Coupling | Output |
|---|---|---|---|---|---|---|---|
| Liu, Ting & Zhou | 2008 | ICDM | UCI tabular | — | Isolation Forest | none | binary score |
| Xu, Pang, Wang & Wang | 2023 | IEEE TKDE | 30+ benchmarks | random / learned MLP per tree | Isolation Forest on learned representation | per-tree projection | binary score |
| Shone, Ngoc, Phai & Shi | 2018 | IEEE TETCI | NSL-KDD, KDD-99 | stacked non-symmetric AEs | Random Forest | sequential (AE → RF) | 5-class label |
| Chen & Guestrin | 2016 | KDD | various | — | XGBoost | none | regression / classification |
| **Phase 3 — Model A** | 2026 | this submission | NSL-KDD | symmetric AE w/ skips | Isolation Forest on AE latent | DL feature extraction → ML decision rule | binary score |
| **Phase 3 — Model C** | 2026 | this submission | NSL-KDD | symmetric AE w/ skips | XGBoost on `[raw ‖ z ‖ residual]` | DL feature extraction + DL error signal → ML supervised classifier; optional AE Stage-1 gate | 5-class label |

## 7. Gap Phase 3 fills

Reading the four anchor papers together, three gaps emerge that Phase 3 closes:

1. **No prior work feeds the autoencoder's *reconstruction residual* into a supervised classifier on NSL-KDD.** Shone et al. feed only the encoded features; Xu et al. feed only the encoded features; conventional AE-based detectors threshold the residual norm but never expose the per-feature residual vector to a downstream learner. Model C's residual block is the architectural innovation, and notebook 04's SHAP analysis empirically verifies that XGBoost relies on it.

2. **No prior work cleanly couples a single trained autoencoder with both a one-class detector (Model A) and a supervised classifier (Model C) on the same evaluation protocol.** Phase 3's notebook 05 ablation does exactly this. The same AE is reused across seven of the eight conditions, eliminating AE-initialisation variance as a confound.

3. **No prior work provides *per-component* diagnostic deltas on NSL-KDD.** Most published hybrids report a single "hybrid > baseline" headline number. The rubric for this submission explicitly rewards diagnostic depth — what does each piece contribute, what happens when each piece is removed. Notebook 05 produces that table for both hybrids end to end.

## 8. Reading the four anchors through the rubric

How each anchor paper informs each rubric component for Phase 3:

| Rubric component | Liu 2008 | Xu 2023 | Shone 2018 | Chen 2016 |
|---|---|---|---|---|
| Hybrid Innovation | Provides the ML decision rule that Model A's hybrid uses to fix Phase-2's calibration weakness. | Provides the architectural template (NN representation + IF) that Model A specialises with a trained AE. | Provides the AE → supervised-classifier template that Model C generalises by adding the residual block. | Justifies the supervised-classifier choice over Random Forest. |
| Ablation Studies | Used as the *raw IF* condition (#2) in the notebook-05 ablation. | Provides the conceptual bound for Model A — our `Deep IF (Model A)` row. | Provides the conceptual bound for Model C without the residual trick — our `XGBoost (raw + latent z)` row. | XGBoost itself is conditions 4–8 in the ablation. |
| Architecture Diagram | The path-length scoring is labelled in `figures/diagram_model_A.png`. | The fusion mechanism (encoder → IF) is drawn explicitly in `figures/diagram_model_A.png`. | The DL → ML data flow is drawn explicitly in `figures/diagram_model_C.png`. | The XGBoost block is named with its specific configuration in `figures/diagram_model_C.png`. |
| Reproducibility | sklearn implementation; deterministic seed. | Re-implemented from scratch in `utils/hybrid_models.DeepIsolationForest`. | Architectural pattern adopted; specific NDAE replaced with our symmetric AE. | XGBoost via `xgboost>=2.0` pinned in `requirements.txt`. |

## 9. Reference list

**Anchor papers cited in this notebook:**

1. Liu, F. T., Ting, K. M., & Zhou, Z.-H. (2008). Isolation Forest. In *Proceedings of the 8th IEEE International Conference on Data Mining (ICDM)*, pp. 413–422. DOI: 10.1109/ICDM.2008.17.

2. Xu, H., Pang, G., Wang, Y., & Wang, Y. (2023). Deep Isolation Forest for Anomaly Detection. *IEEE Transactions on Knowledge and Data Engineering*. DOI: 10.1109/TKDE.2023.3270293.

3. Shone, N., Ngoc, T. N., Phai, V. D., & Shi, Q. (2018). A Deep Learning Approach to Network Intrusion Detection. *IEEE Transactions on Emerging Topics in Computational Intelligence*, 2(1), 41–50. DOI: 10.1109/TETCI.2017.2772792.

4. Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. In *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, pp. 785–794. DOI: 10.1145/2939672.2939785.

**Background references mentioned in passing (no per-section analysis):**

5. Tavallaee, M., Bagheri, E., Lu, W., & Ghorbani, A. A. (2009). A Detailed Analysis of the KDD CUP 99 Data Set. In *IEEE Symposium on Computational Intelligence for Security and Defense Applications (CISDA)*. — NSL-KDD dataset paper.

6. Sakurada, M., & Yairi, T. (2014). Anomaly Detection Using Autoencoders with Nonlinear Dimensionality Reduction. In *MLSDA 2014*. — Foundational autoencoder-anomaly-detection reference (already cited in Phase 2).

7. Chandola, V., Banerjee, A., & Kumar, V. (2009). Anomaly Detection: A Survey. *ACM Computing Surveys*, 41(3), Article 15. — Canonical taxonomy of anomaly-detection paradigms.

8. Buczak, A. L., & Guven, E. (2016). A Survey of Data Mining and Machine Learning Methods for Cyber Security Intrusion Detection. *IEEE Communications Surveys & Tutorials*, 18(2), 1153–1176. — Modern survey of IDS-specific ML methods.

9. Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. *Annals of Statistics*, 29(5), 1189–1232. — Foundation paper for gradient boosting that XGBoost extends.

10. Erfani, S. M., Rajasegarar, S., Karunasekera, S., & Leckie, C. (2016). High-dimensional and large-scale anomaly detection using a linear one-class SVM with deep learning. *Pattern Recognition*, 58, 121–134. — Alternative anchor for Model A; cited here as suggested-alternative reading.

11. Javaid, A., Niyaz, Q., Sun, W., & Alam, M. (2016). A Deep Learning Approach for Network Intrusion Detection System. In *Proceedings of EAI BICT 2016*. — Alternative anchor for Model C; cited here as suggested-alternative reading.

## 10. Hand-off

Notebook 02 (`02_data_preparation.ipynb`) operationalises the leakage controls discussed in Section 3 (per-class imbalance, distribution shift) and produces the splits that every downstream notebook consumes. The four anchor papers are cited again, in the right places, throughout notebooks 03–06.